# Satellite close-approach hackathon

## Aim of this exercise

You are given a **catalog of satellites** (orbital data at known times). Your job is to write an algorithm that finds **possible collisions**: close approaches — pairs of satellites that come near each other while you **propagate** (calculate) their positions along the orbit forward in time.

### What you are given

| Item | What it is |
|------|------------|
| `spacetrack_data.json` | Catalog of ~17k objects whose orbit data (epoch) is from **1 June 2026** |
| `conjunction_toolkit/` | Library to load the catalog, propagate positions, run a naive baseline, plot, and verify claims |
| This notebook | Walkthrough + verifier. **Your code goes directly in a cell below** — no external file needed. |

Each catalog row includes an object id, name, epoch (when the orbit was measured), and orbital elements used by SGP4 to predict future positions.

### End goal

1. Implement `find_close_approaches` in the **"Your solution" cell** further down.
2. For a chosen time range (`propagate_from_utc` → `propagate_until_utc`), return every pair that comes within a distance threshold.
3. Each answer is a **claim**: two object ids + time of closest approach + miss distance (km).
4. Pass the shared **verifier**, which re-propagates both objects and checks your time and distance.

Beat the **naive baseline** (check every pair on a coarse time grid) on speed and/or quality of verified claims.

#### Winning team

The team that finds as many collisions as possible in a maximum of **5 minutes** of runtime. If two teams find the same number of collisions, the **faster** team wins.

`MAX_RUNTIME_SECONDS` controls the time limit for both the naive baseline and your algorithm (default **300 s = 5 minutes**).


## 0. Setup (Google Colab)

Run the next cell once (or **Runtime → Run all**). It installs packages and downloads the catalog and toolkit from GitHub. No external file needed for your solution — your code lives in this notebook.


In [ ]:
import os
import sys
import shutil
import subprocess
import time
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

pkgs = [
    "skyfield>=1.48",
    "sgp4>=2.23",
    "numpy>=1.26",
    "scipy>=1.11",
    "plotly>=5.18",
    "pandas>=2.1",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

CONTENT = Path("/content") if Path("/content").exists() else Path.cwd()
ZIP_PATH = CONTENT / "solve_branch.zip"
ROOT = CONTENT / "hackathon"
# Always re-download so Colab gets the latest solve branch (catalog + toolkit + student_solution.py)
ZIP_URLS = [
    f"https://codeload.github.com/abensour/collision_detection_hackathon/zip/refs/heads/solve?t={int(time.time())}",
    f"https://github.com/abensour/collision_detection_hackathon/archive/refs/heads/solve.zip?t={int(time.time())}",
]


def _download(url: str, dest: Path) -> None:
    print("Downloading", url.split("?")[0], "…")
    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(req, timeout=180) as resp, dest.open("wb") as out:
        shutil.copyfileobj(resp, out)


last_error = None
for url in ZIP_URLS:
    try:
        _download(url, ZIP_PATH)
        last_error = None
        break
    except Exception as exc:
        last_error = exc
        print("Download failed:", exc)
if last_error is not None:
    raise RuntimeError("Could not download the student files from GitHub") from last_error

zip_bytes = ZIP_PATH.stat().st_size
print(f"Downloaded {zip_bytes / 1e6:.2f} MB")
if zip_bytes < 500_000:
    raise RuntimeError(f"Zip is too small ({zip_bytes} bytes) — download did not get the catalog")

extract_parent = CONTENT / "_hackathon_extract"
if extract_parent.exists():
    shutil.rmtree(extract_parent)
extract_parent.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(extract_parent)
    names = zf.namelist()

inner_dirs = [p for p in extract_parent.iterdir() if p.is_dir()]
inner = inner_dirs[0] if len(inner_dirs) == 1 else extract_parent

if ROOT.exists():
    shutil.rmtree(ROOT)
shutil.copytree(inner, ROOT)
shutil.rmtree(extract_parent, ignore_errors=True)

required = [
    ROOT / "spacetrack_data.json",
    ROOT / "conjunction_toolkit" / "__init__.py",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing from zip: " + ", ".join(missing))

catalog_bytes = (ROOT / "spacetrack_data.json").stat().st_size
print(f"Catalog spacetrack_data.json: {catalog_bytes / 1e6:.2f} MB")
if catalog_bytes < 1_000_000:
    raise RuntimeError("Catalog file is too small — data did not download")

print("Files in /content/hackathon:")
for path in sorted(ROOT.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".ipynb":
        print(" ", path.relative_to(ROOT))

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Ready. Project root:", ROOT)
print("Your algorithm goes in the 'Your solution' cell in this notebook.")


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timedelta
from itertools import combinations
from typing import Callable, Dict, List
import time

from skyfield.api import EarthSatellite

from conjunction_toolkit import (
    ConjunctionClaim,
    catalog_to_satellites,
    closest_approach_on_grid,
    load_default_catalog,
    plot_pair_with_distance,
    plot_trajectories,
    propagate_many,
    save_html,
    time_grid,
)
from conjunction_toolkit.propagate import datetimes_of

OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Toolkit cheat sheet (what you may call)

| Function | What it does |
|----------|----------------|
| `load_default_catalog()` | Load `spacetrack_data.json` |
| `catalog_to_satellites(catalog, norad_ids=...)` | Build objects you can propagate |
| `time_grid(from, until, step_seconds)` | Sample times from start **until** end |
| `propagate_many(satellites, times)` | Positions (km) for many satellites |
| `closest_approach_on_grid(pos_a, pos_b, times)` | Among samples, when were two objects nearest? |
| `verify_claim(claim, satellites)` | Independent check of one claim |
| `plot_trajectories` / `plot_pair_with_distance` / `save_html` | Plots |

**Claim fields:** `norad_a`, `norad_b`, `tca_utc` (claimed time in UTC). Optional: `min_distance_km`, `algorithm_id`.


## 2. Load the catalog

In [ ]:
catalog = load_default_catalog()

print(f"Source  : {catalog.source_path}")
print(f"Objects : {len(catalog)}")

epochs = [obj.epoch_utc for obj in catalog]
print(f"Epochs  : {min(epochs).date()} → {max(epochs).date()}  (data dates, not your propagate range)")

print("\nSample objects:")
for object_id in catalog.ids()[:5]:
    obj = catalog[object_id]
    print(f"  id={obj.norad_cat_id:>6}  {obj.object_name:<28}  epoch={obj.epoch_utc.date()}")

## 3. Propagate and plot a few orbits

Note that in the plot the sphere does not rotate, whereas in reality Earth does.


In [ ]:
demo_ids: list[int] = []
for name in ("ISS (ZARYA)", "HST", "CSS (TIANHE)"):
    matches = list(catalog.filter_by_name(name))
    if matches:
        matches.sort(key=lambda obj: obj.epoch_utc, reverse=True)
        demo_ids.append(matches[0].norad_cat_id)

if len(demo_ids) < 2:
    recent = sorted(catalog, key=lambda obj: obj.epoch_utc, reverse=True)
    demo_ids = [obj.norad_cat_id for obj in recent[:3]]

demo_ids = demo_ids[:3]
demo_satellites = catalog_to_satellites(catalog, norad_ids=demo_ids)

propagate_from_utc = max(catalog[i].epoch_utc for i in demo_ids)
PROPAGATE_FOR_HOURS = 6
propagate_until_utc = propagate_from_utc + timedelta(hours=PROPAGATE_FOR_HOURS)
orbit_times = time_grid(propagate_from_utc, propagate_until_utc, step_seconds=60)

print("Objects:", [(i, demo_satellites[i].name) for i in demo_ids])
print(f"Propagate from : {propagate_from_utc.isoformat()}")
print(f"Propagate until: {propagate_until_utc.isoformat()}  ({PROPAGATE_FOR_HOURS} hours ahead)")

fig = plot_trajectories(demo_satellites, orbit_times, title="Sample orbits")
save_html(fig, OUTPUT_DIR / "notebook_trajectories.html")
fig.show()

## 4. Naive baseline — runs on ALL satellites, stops after 30 seconds

This is the **reference algorithm** you need to beat. It runs on the **full catalog** (~17k objects) but is **hard-capped at 30 seconds** — the O(N²) pair loop over 17k objects would otherwise take hours.

What it does:

1. Loads **all** satellites from the catalog.
2. Builds sample times from `propagate_from_utc` until `propagate_until_utc`.
3. Propagates every satellite to those times.
4. Iterates over **every unique pair** (≈148 million for 17k objects), stopping as soon as 30 s have elapsed.

| Setting | Value | Meaning |
|---------|-------|---------|
| `PROPAGATE_FOR_HOURS` | **6** | How far ahead to search |
| `TIME_STEP_MINUTES` | **2** | Example step used by the naive baseline |
| `CLOSE_APPROACH_THRESHOLD_KM` | **100** | Keep pairs within 100 km |
| `NAIVE_RUNTIME_SECONDS` | **30** | Hard stop for the naive baseline |
| `MAX_RUNTIME_SECONDS` | **300** | Time limit passed to **your** `find_close_approaches` |

> **Choosing a time step is your design decision.** The 2-minute value above is just a demo. Smaller steps find more events; larger steps are faster but can miss brief close approaches. You control `time_step_seconds` in your own algorithm.


In [ ]:
# ── Shared settings ──────────────────────────────────────────────────────────
PROPAGATE_FOR_HOURS = 6
CLOSE_APPROACH_THRESHOLD_KM = 100.0
MAX_RUNTIME_SECONDS = 300   # 5 minutes — time limit passed to your find_close_approaches

# ── Naive baseline settings ───────────────────────────────────────────────────
# TIME_STEP_MINUTES is an EXAMPLE value for the naive baseline only.
# For your own solution, choosing the right time step is part of the challenge:
#   - smaller step  →  more accurate, slower
#   - larger step   →  faster, but can miss short close approaches
TIME_STEP_MINUTES = 2
TIME_STEP_SECONDS = TIME_STEP_MINUTES * 60

NAIVE_RUNTIME_SECONDS = 30  # naive baseline hard-stops after 30 s (too slow for full catalog)

# ── Load ALL satellites from the catalog ──────────────────────────────────────
# catalog was loaded in the cell above; re-use it here
all_ids = catalog.ids()
satellites = catalog_to_satellites(catalog, norad_ids=all_ids)

epochs = sorted(catalog[nid].epoch_utc for nid in all_ids)
propagate_from_utc = epochs[len(epochs) // 2]
propagate_until_utc = propagate_from_utc + timedelta(hours=PROPAGATE_FOR_HOURS)

n_pairs = len(all_ids) * (len(all_ids) - 1) // 2
n_samples = int(PROPAGATE_FOR_HOURS * 3600 / TIME_STEP_SECONDS) + 1

print(f"Catalog loaded: {len(all_ids)} satellites")
print(f"Pairs to check (naive, full): {n_pairs:,}")
print(f"Propagate from : {propagate_from_utc.isoformat()}")
print(f"Propagate until: {propagate_until_utc.isoformat()}  ({PROPAGATE_FOR_HOURS} h ahead)")
print(f"Naive time step: {TIME_STEP_MINUTES} min  (example only — pick your own step in your solution)")
print(f"Naive hard stop: {NAIVE_RUNTIME_SECONDS} s")
print(f"Your time limit: {MAX_RUNTIME_SECONDS} s")


In [ ]:
def naive_baseline_find_close_approaches(
    satellites,
    propagate_from_utc,
    propagate_until_utc,
    time_step_seconds: float,
    close_approach_threshold_km: float,
    max_runtime_seconds: float,
):
    """Naive O(N²) baseline: check every unique pair on a regular time grid.

    Runs on ALL satellites but stops after max_runtime_seconds.
    With ~17k satellites the pair loop (≈148M pairs) can never finish —
    that is intentional: this is the baseline you need to beat.
    """
    object_ids = sorted(satellites.keys())
    if len(object_ids) < 2:
        return []

    # Sample times
    sample_skyfield_times = time_grid(
        propagate_from_utc, propagate_until_utc, time_step_seconds,
    )
    sample_times = datetimes_of(sample_skyfield_times)

    # Propagate everyone
    positions_by_id = propagate_many(satellites, sample_skyfield_times)

    # Check every unique pair — stops when time runs out
    claims = []
    pairs_checked = 0
    t_start = time.perf_counter()
    for id_a, id_b in combinations(object_ids, 2):
        if time.perf_counter() - t_start >= max_runtime_seconds:
            elapsed = time.perf_counter() - t_start
            print(
                f"  ⏱  Naive baseline stopped at {elapsed:.1f} s "
                f"after checking {pairs_checked:,} / {len(object_ids)*(len(object_ids)-1)//2:,} pairs "
                f"(this is expected — the full O(N²) loop would take hours)."
            )
            break
        closest_time, closest_distance_km, _ = closest_approach_on_grid(
            positions_by_id[id_a], positions_by_id[id_b], sample_times,
        )
        pairs_checked += 1
        if closest_distance_km <= close_approach_threshold_km:
            claims.append(ConjunctionClaim(
                norad_a=id_a, norad_b=id_b,
                tca_utc=closest_time, min_distance_km=closest_distance_km,
                algorithm_id="naive_baseline",
            ))

    claims.sort(key=lambda c: float("inf") if c.min_distance_km is None else c.min_distance_km)
    return claims


print("Running naive baseline on ALL satellites (stops after 30 s)…")
t0 = time.perf_counter()
baseline_raw = naive_baseline_find_close_approaches(
    satellites,
    propagate_from_utc,
    propagate_until_utc,
    TIME_STEP_SECONDS,
    CLOSE_APPROACH_THRESHOLD_KM,
    NAIVE_RUNTIME_SECONDS,   # hard 30-second cap
)
elapsed = time.perf_counter() - t0
print(f"Done in {elapsed:.2f} s  |  Close approaches found in that window: {len(baseline_raw)}")
if baseline_raw:
    print("Closest few:")
    for claim in baseline_raw[:5]:
        dist = "n/a" if claim.min_distance_km is None else f"{claim.min_distance_km:.3f} km"
        print(f"  {claim.norad_a}–{claim.norad_b}: {dist} at {claim.tca_utc.isoformat()}")


## 5. Your solution — implement `find_close_approaches` in the cell below

The function stub is right below. Fill it in — **no external file needed**, your code lives here.

```python
def find_close_approaches(
    satellites,            # dict: NORAD id → EarthSatellite
    propagate_from_utc,    # start time (UTC datetime)
    propagate_until_utc,   # end time   (UTC datetime)
    time_step_seconds,     # your choice — see note below
    close_approach_threshold_km,
    max_runtime_seconds,   # hard stop: return what you have when time is up
) -> list[ConjunctionClaim]:
    ...
```

> **Choosing `time_step_seconds`:** This is part of the problem. The verifier passes the same value you used, so the step is yours to decide. Smaller steps find more events but take longer. A good starting point is anywhere from 30 s to 5 minutes depending on your approach.

The verifier (section 6) calls this function and compares it to the naive baseline.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# YOUR SOLUTION — implement the function below
# ══════════════════════════════════════════════════════════════════

def find_close_approaches(
    satellites: Dict[int, EarthSatellite],
    propagate_from_utc: datetime,
    propagate_until_utc: datetime,
    time_step_seconds: float,        # your choice — smaller = more accurate, slower
    close_approach_threshold_km: float,
    max_runtime_seconds: float,      # stop and return what you have when this elapses
) -> List[ConjunctionClaim]:
    """Find pairs of satellites that come within close_approach_threshold_km.

    Parameters
    ----------
    satellites:
        Dict mapping NORAD catalog number → Skyfield EarthSatellite.
    propagate_from_utc, propagate_until_utc:
        Search window. Propagate orbits between these two UTC datetimes.
    time_step_seconds:
        How often to sample positions. Choosing a good value is part of the
        challenge — coarser steps are faster but can miss short encounters.
    close_approach_threshold_km:
        Report a pair only if their distance at your claimed time ≤ this value.
    max_runtime_seconds:
        Hard time limit. Stop searching and return current claims when elapsed.

    Returns
    -------
    list[ConjunctionClaim]
        One entry per close pair found. Each claim needs:
        - norad_a, norad_b : the two NORAD ids
        - tca_utc          : claimed time of the conjunction (UTC datetime)

        Optional:
        - min_distance_km  : distance in km if you computed it
        - algorithm_id     : short label for your method
    """
    raise NotImplementedError("Implement your algorithm here.")


## 6. Verifier (simple True/Failed at claimed time)

For each claim, the verifier checks only this:

1. Take the student-provided pair (`norad_a`, `norad_b`)
2. Propagate both satellites at the student-provided time (`tca_utc`)
3. Compute distance at that exact time
4. Mark:
   - **True** if distance <= threshold
   - **Failed** otherwise

This means we do **not** search for the real closest time. We only test whether there is really a conjunction at the time reported by the student.


In [ ]:
@dataclass
class EvaluationReport:
    algorithm_name: str
    close_approaches_detected: int
    claims_verified_ok: int
    claims_failed: int
    runtime_seconds: float
    messages: List[str] = field(default_factory=list)
    claims: List[ConjunctionClaim] = field(default_factory=list)

    def print_summary(self) -> None:
        print(f"=== {self.algorithm_name} ===")
        print(f"  Close approaches detected : {self.close_approaches_detected}")
        print(f"  Verified True             : {self.claims_verified_ok}")
        print(f"  Failed                    : {self.claims_failed}")
        print(f"  Runtime                   : {self.runtime_seconds:.2f} s")
        for msg in self.messages:
            print(f"  - {msg}")


def verify_claim_at_reported_time(
    claim: ConjunctionClaim,
    satellites: Dict[int, EarthSatellite],
    close_approach_threshold_km: float,
) -> tuple[bool, float, str]:
    sat_a = satellites.get(claim.norad_a)
    sat_b = satellites.get(claim.norad_b)
    if sat_a is None or sat_b is None:
        missing = []
        if sat_a is None:
            missing.append(str(claim.norad_a))
        if sat_b is None:
            missing.append(str(claim.norad_b))
        return False, float('nan'), f"Missing satellites in catalog: {', '.join(missing)}"

    try:
        t = time_grid(claim.tca_utc, claim.tca_utc, step_seconds=1.0)
        times = datetimes_of(t)
        local_sats = {claim.norad_a: sat_a, claim.norad_b: sat_b}
        positions = propagate_many(local_sats, t)
        _, distance_km, _ = closest_approach_on_grid(
            positions[claim.norad_a],
            positions[claim.norad_b],
            times,
        )
    except Exception as exc:
        return False, float('nan'), f"Propagation failed: {exc}"

    if distance_km <= close_approach_threshold_km:
        return True, distance_km, "True"
    return False, distance_km, (
        f"Failed: distance at claimed time is {distance_km:.6f} km "
        f"> threshold {close_approach_threshold_km:.6f} km"
    )


def evaluate_finder(
    find_close_approaches: Callable[..., List[ConjunctionClaim]],
    satellites: Dict[int, EarthSatellite],
    propagate_from_utc: datetime,
    propagate_until_utc: datetime,
    time_step_seconds: float,
    close_approach_threshold_km: float,
    algorithm_name: str,
    max_runtime_seconds: float = 300.0,
) -> EvaluationReport:
    t0 = time.perf_counter()
    raw_claims = find_close_approaches(
        satellites,
        propagate_from_utc,
        propagate_until_utc,
        time_step_seconds=time_step_seconds,
        close_approach_threshold_km=close_approach_threshold_km,
        max_runtime_seconds=max_runtime_seconds,
    )
    runtime_seconds = time.perf_counter() - t0

    claims = list(raw_claims)
    ok_count = 0
    fail_count = 0
    messages: List[str] = []

    for claim in claims:
        ok, distance_km, message = verify_claim_at_reported_time(
            claim,
            satellites,
            close_approach_threshold_km,
        )
        if ok:
            ok_count += 1
        else:
            fail_count += 1
            messages.append(
                f"FAIL {claim.norad_a}–{claim.norad_b} @ {claim.tca_utc.isoformat()} | {message}"
            )

    return EvaluationReport(
        algorithm_name=algorithm_name,
        close_approaches_detected=len(raw_claims),
        claims_verified_ok=ok_count,
        claims_failed=fail_count,
        runtime_seconds=runtime_seconds,
        messages=messages,
        claims=claims,
    )


print("Evaluator ready (claimed-time check).")


In [ ]:
baseline_report = evaluate_finder(
    naive_baseline_find_close_approaches,
    satellites,
    propagate_from_utc,
    propagate_until_utc,
    TIME_STEP_SECONDS,
    CLOSE_APPROACH_THRESHOLD_KM,
    "naive_baseline",
    max_runtime_seconds=NAIVE_RUNTIME_SECONDS,  # 30 s hard stop
)
baseline_report.print_summary()


In [ ]:
# Re-run this cell every time you change your find_close_approaches above

try:
    student_report = evaluate_finder(
        find_close_approaches,      # the function defined in the cell above
        satellites,
        propagate_from_utc,
        propagate_until_utc,
        TIME_STEP_SECONDS,
        CLOSE_APPROACH_THRESHOLD_KM,
        "my_solution",
        max_runtime_seconds=MAX_RUNTIME_SECONDS,
    )
    student_report.print_summary()
except NotImplementedError as exc:
    print("Implement find_close_approaches in the cell above first.")
    student_report = None


In [ ]:
if student_report is not None:
    print(f"{'algorithm':<20} {'detected':>10} {'verified_ok':>12} {'failed':>8} {'seconds':>10}")
    for report in (baseline_report, student_report):
        print(
            f"{report.algorithm_name:<20} "
            f"{report.close_approaches_detected:>10} "
            f"{report.claims_verified_ok:>12} "
            f"{report.claims_failed:>8} "
            f"{report.runtime_seconds:>10.2f}"
        )
else:
    print("student_solution.py did not run. Re-run Setup, then this cell.")


### Optional: plot one close approach

In [ ]:
report_to_plot = student_report if student_report is not None else baseline_report

if report_to_plot.claims:
    best = report_to_plot.claims[0]
    zoom = time_grid(
        best.tca_utc - timedelta(minutes=45),
        best.tca_utc + timedelta(minutes=45),
        step_seconds=30.0,
    )
    fig = plot_pair_with_distance(
        satellites[best.norad_a],
        satellites[best.norad_b],
        zoom,
        tca=best.tca_utc,
        norad_a=best.norad_a,
        norad_b=best.norad_b,
        threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
        title=f"Close approach {best.norad_a}–{best.norad_b}",
    )
    save_html(fig, OUTPUT_DIR / "notebook_pair.html")
    fig.show()
else:
    print("No claims to plot.")
